#Question 1 — Iris Dataset Statistical Analysis

---



##Task 1 — Loading and Visualising the Iris Dataset

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
iris = sns.load_dataset('iris')
fig, axes = plt.subplots(2,3, figsize=(12,8))
pairs = [('sepal_length','sepal_width'),('sepal_length','petal_length'),('sepal_length','petal_width'),
         ('sepal_width','petal_length'),('sepal_width','petal_width'),('petal_length','petal_width')]
for ax, (x,y) in zip(axes.flat, pairs):
    for sp in iris['species'].unique():
        sub = iris[iris.species==sp]
        ax.scatter(sub[x], sub[y], label=sp, alpha=0.7, s=20)
    ax.set_xlabel(x); ax.set_ylabel(y)
plt.tight_layout(); plt.show()
print('Setosa is clearly separable from the other two species across all feature pairs. Versicolor and Virginica overlap partially, especially in sepal measurements.')

Setosa is clearly separable from the other two species across all feature pairs. Versicolor and Virginica overlap partially, especially in sepal measurements.


##Task 2 — Descriptive Statistics and Distribution Analysis

In [ ]:
from scipy.stats import skew, kurtosis
features = ['sepal_length','sepal_width','petal_length','petal_width']
print('=== Overall ===')
print(iris[features].describe())
for f in features:
    print(f'{f}: skew={skew(iris[f]):.2f}, kurtosis={kurtosis(iris[f]):.2f}')
for sp in iris['species'].unique():
    print(f'--- {sp} ---')
    print(iris[iris.species==sp][features].describe())
    for f in features:
        col = iris[iris.species==sp][f]
        print(f'{f}: skew={skew(col):.2f}, kurtosis={kurtosis(col):.2f}')
fig, axes = plt.subplots(2,2, figsize=(10,8))
for ax, f in zip(axes.flat, features):
    iris[f].plot(kind='hist', bins=8, density=True, ax=ax, alpha=0.6)
    iris[f].plot(kind='kde', ax=ax, secondary_y=True, color='red')
    ax.set_title(f)
plt.tight_layout(); plt.show()
print('Sepal width is roughly symmetric. Petal features are bimodal and left-skewed overall due to species differences.')

=== Overall ===


       sepal_length  sepal_width  petal_length  petal_width
count    150.000000   150.000000    150.000000   150.000000
mean       5.843333     3.057333      3.758000     1.199333
std        0.828066     0.435866      1.765298     0.762238
min        4.300000     2.000000      1.000000     0.100000
25%        5.100000     2.800000      1.600000     0.300000
50%        5.800000     3.000000      4.350000     1.300000
75%        6.400000     3.300000      5.100000     1.800000
max        7.900000     4.400000      6.900000     2.500000

sepal_length: skew=0.31, kurtosis=-0.55
sepal_width: skew=0.31, kurtosis=0.14
petal_length: skew=-0.27, kurtosis=-1.40
petal_width: skew=-0.10, kurtosis=-1.34
--- setosa ---
       sepal_length  sepal_width  petal_length  petal_width
count     50.00000    50.00000      50.00000    50.000000
mean       5.00600     3.42800       1.46200     0.246000
std        0.35249     0.379064      0.173664    0.105386
min        4.30000     2.30000       1.00000     0.

Sepal width is roughly symmetric. Petal features are bimodal and left-skewed overall due to species differences.


##Task 3 — ANOVA and Confidence Intervals

In [ ]:
from scipy.stats import f_oneway, t
import numpy as np
for f in features:
    groups = [iris[iris.species==sp][f] for sp in iris['species'].unique()]
    f_stat, p_val = f_oneway(*groups)
    print(f'{f}: F={f_stat:.2f}, p={p_val:.2e} | Significant: {"Yes" if p_val<0.05 else "No"}')
print()
for f in features:
    print(f'--- {f} 95% CI ---')
    for sp in iris['species'].unique():
        col = iris[iris.species==sp][f]
        mean = col.mean()
        se = col.std() / np.sqrt(len(col))
        ci = t.ppf(0.975, len(col)-1) * se
        print(f'  {sp}: {mean:.2f} +/- {ci:.2f}  ({mean-ci:.2f}, {mean+ci:.2f})')
print()
print('ANOVA confirms highly significant differences across species for all four features (p \u2248 0).')

sepal_length: F=119.26, p=1.67e-31 | Significant: Yes
sepal_width: F=49.16, p=4.49e-17 | Significant: Yes
petal_length: F=1180.16, p=2.86e-91 | Significant: Yes
petal_width: F=960.01, p=4.17e-85 | Significant: Yes

--- sepal_length 95% CI ---
  setosa: 5.01 +/- 0.10  (4.91, 5.11)
  versicolor: 5.94 +/- 0.15  (5.79, 6.08)
  virginica: 6.59 +/- 0.18  (6.41, 6.77)
--- sepal_width 95% CI ---
  setosa: 3.43 +/- 0.11  (3.32, 3.54)
  versicolor: 2.77 +/- 0.09  (2.68, 2.86)
  virginica: 2.97 +/- 0.09  (2.88, 3.07)
--- petal_length 95% CI ---
  setosa: 1.46 +/- 0.05  (1.41, 1.51)
  versicolor: 4.26 +/- 0.13  (4.13, 4.39)
  virginica: 5.55 +/- 0.16  (5.40, 5.71)
--- petal_width 95% CI ---
  setosa: 0.25 +/- 0.03  (0.22, 0.28)
  versicolor: 1.33 +/- 0.06  (1.27, 1.38)
  virginica: 2.03 +/- 0.08  (1.95, 2.10)

ANOVA confirms highly significant differences across species for all four features (p ≈ 0).


##Task 4 — Pairwise T-Tests with Correction

In [ ]:
from scipy.stats import ttest_ind
species = iris['species'].unique()
pairs_list = [('setosa','versicolor'),('setosa','virginica'),('versicolor','virginica')]
for f in features:
    print(f'--- {f} ---')
    for s1, s2 in pairs_list:
        g1 = iris[iris.species==s1][f]; g2 = iris[iris.species==s2][f]
        t_stat, p_raw = ttest_ind(g1, g2)
        p_corrected = min(p_raw * 3, 1.0)
        sig = 'Sig' if p_corrected < 0.05 else 'NOT sig'
        print(f'  {s1} vs {s2}: t={t_stat:.2f}, p_corrected={p_corrected:.2e} [{sig}]')
print()
print('Setosa is significantly different from both other species for all features. Versicolor vs Virginica differ significantly in all features.')

--- sepal_length ---
  setosa vs versicolor: t=-10.52, p_corrected=2.70e-17 [Sig]
  setosa vs virginica: t=-15.39, p_corrected=2.07e-27 [Sig]
  versicolor vs virginica: t=-5.63, p_corrected=5.17e-07 [Sig]
--- sepal_width ---
  setosa vs versicolor: t=9.45, p_corrected=5.54e-15 [Sig]
  setosa vs virginica: t=6.45, p_corrected=1.27e-08 [Sig]
  versicolor vs virginica: t=-3.21, p_corrected=5.46e-03 [Sig]
--- petal_length ---
  setosa vs versicolor: t=-39.49, p_corrected=1.62e-61 [Sig]
  setosa vs virginica: t=-49.99, p_corrected=4.51e-71 [Sig]
  versicolor vs virginica: t=-12.60, p_corrected=9.54e-22 [Sig]
--- petal_width ---
  setosa vs versicolor: t=-34.08, p_corrected=1.15e-55 [Sig]
  setosa vs virginica: t=-42.79, p_corrected=9.69e-65 [Sig]
  versicolor vs virginica: t=-14.63, p_corrected=6.69e-26 [Sig]

Setosa is significantly different from both other species for all features. Versicolor vs Virginica differ significantly in all features.
